# FASE2-P1 — Data Foundation

**Objetivo:** Carregar os 9 CSVs da Olist, pre-agregar geolocation, construir o join chain principal e produzir `gold_with_geo` com 1 linha por order_id.  
**Ancora temporal:** `order_approved_at` (nenhuma variavel pos-entrega no modelo)  
**Output:** DataFrame `gold_with_geo` — base da tabela gold para Phases 3 e 4.

## Secao 1: Setup e Load

In [1]:
# Celula 1: imports
from pathlib import Path
import pandas as pd
import numpy as np
import sys

# Project root detection — works whether executed from notebooks/ or project root
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == 'notebooks':
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

DATA_RAW  = PROJECT_ROOT / 'data' / 'raw'
DATA_GOLD = PROJECT_ROOT / 'data' / 'gold'
DATA_GOLD.mkdir(parents=True, exist_ok=True)

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'DATA_RAW exists: {DATA_RAW.exists()}')
print(f'DATA_GOLD: {DATA_GOLD}')

PROJECT_ROOT: C:\Users\Wey\Desktop\Alpha\Projetos\python
DATA_RAW exists: True
DATA_GOLD: C:\Users\Wey\Desktop\Alpha\Projetos\python\data\gold


In [2]:
# Celula 2: carregar os 9 CSVs com parse_dates onde aplicavel
orders = pd.read_csv(DATA_RAW / 'olist_orders_dataset.csv',
    parse_dates=['order_purchase_timestamp', 'order_approved_at',
                 'order_delivered_carrier_date', 'order_delivered_customer_date',
                 'order_estimated_delivery_date'])
items    = pd.read_csv(DATA_RAW / 'olist_order_items_dataset.csv')
reviews  = pd.read_csv(DATA_RAW / 'olist_order_reviews_dataset.csv',
    parse_dates=['review_creation_date', 'review_answer_timestamp'])
payments = pd.read_csv(DATA_RAW / 'olist_order_payments_dataset.csv')
customers = pd.read_csv(DATA_RAW / 'olist_customers_dataset.csv')
sellers   = pd.read_csv(DATA_RAW / 'olist_sellers_dataset.csv')
products  = pd.read_csv(DATA_RAW / 'olist_products_dataset.csv')
geo       = pd.read_csv(DATA_RAW / 'olist_geolocation_dataset.csv')
cat_trans = pd.read_csv(DATA_RAW / 'product_category_name_translation.csv')

print('Todos os 9 CSVs carregados com sucesso.')

Todos os 9 CSVs carregados com sucesso.


In [3]:
# Celula 3: validacao de shapes — printar shape e primeiras linhas de cada df
shapes = {
    'orders':    orders.shape,
    'items':     items.shape,
    'reviews':   reviews.shape,
    'payments':  payments.shape,
    'customers': customers.shape,
    'sellers':   sellers.shape,
    'products':  products.shape,
    'geo':       geo.shape,
    'cat_trans': cat_trans.shape,
}
for name, shape in shapes.items():
    print(f'{name}: {shape}')
# Esperado: orders ~(99441, 8), items ~(112650, 7), reviews ~(99224, 7)

orders: (99441, 8)
items: (112650, 7)
reviews: (99224, 7)
payments: (103886, 5)
customers: (99441, 5)
sellers: (3095, 4)
products: (32951, 9)
geo: (1000163, 5)
cat_trans: (71, 2)


## Secao 2: Pre-agregacao da Geolocation (CRITICO)

A tabela `geo` tem MULTIPLAS linhas por `zip_code_prefix`.  
Se fizermos join direto sem pre-agregacao, o resultado explodiria a cardinalidade.  
Solucao: agregar com `mean()` de lat e lon antes de qualquer join.

In [4]:
# Celula 4: pre-agregar geolocation para 1 linha por zip_code_prefix
# geo tem MULTIPLAS linhas por prefixo — usar mean() para lat e lon
geo_agg = (
    geo.groupby('geolocation_zip_code_prefix')
    .agg(
        geolocation_lat=('geolocation_lat', 'mean'),
        geolocation_lng=('geolocation_lng', 'mean'),
    )
    .reset_index()
)
print(f'geo original: {geo.shape[0]} linhas')
print(f'geo_agg (1 linha por prefix): {geo_agg.shape[0]} linhas')
# Validar: geo_agg deve ter significativamente menos linhas que geo
assert geo_agg['geolocation_zip_code_prefix'].is_unique, 'zip prefix deve ser unico apos agregacao'

geo original: 1000163 linhas
geo_agg (1 linha por prefix): 19015 linhas


## Secao 3: Agregacoes de Items, Reviews e Payments

In [5]:
# Celula 5: agregar items por order_id (1 pedido pode ter multiplos itens)
items_agg = (
    items.groupby('order_id')
    .agg(
        freight_value=('freight_value', 'sum'),
        payment_value=('price', 'sum'),   # preco total dos itens
        seller_id=('seller_id', 'first'),  # seller principal do pedido
        product_id=('product_id', 'first'),
    )
    .reset_index()
)
print(f'items_agg shape: {items_agg.shape}')
assert items_agg['order_id'].is_unique

items_agg shape: (98666, 5)


In [6]:
# Celula 6: desduplicar reviews (alguns pedidos tem multiplas reviews — pegar a mais recente)
reviews_dedup = (
    reviews.sort_values('review_answer_timestamp')
    .drop_duplicates(subset='order_id', keep='last')
    [['order_id', 'review_score']]
)
print(f'reviews original: {reviews.shape[0]} | apos dedup: {reviews_dedup.shape[0]}')

reviews original: 99224 | apos dedup: 98673


In [7]:
# Celula 7: agregar payments por order_id (pedidos podem ter multiplos metodos)
payments_agg = (
    payments.groupby('order_id')
    .agg(total_payment_value=('payment_value', 'sum'))
    .reset_index()
)

## Secao 4: Join Chain Principal

A tabela `orders` e a ancora. Todos os joins sao `how='left'` para preservar todos os pedidos.  
REGRA: nenhuma coluna pos-entrega entra na tabela gold.

In [8]:
# Celula 8: join em sequencia — orders como ancora
gold_raw = (
    orders
    .merge(items_agg, on='order_id', how='left')
    .merge(reviews_dedup, on='order_id', how='left')
    .merge(payments_agg, on='order_id', how='left')
    .merge(customers[['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
                       'customer_city', 'customer_state']], on='customer_id', how='left')
    .merge(sellers[['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']],
           on='seller_id', how='left')
    .merge(products[['product_id', 'product_category_name', 'product_weight_g',
                     'product_length_cm', 'product_height_cm', 'product_width_cm',
                     'product_photos_qty']], on='product_id', how='left')
    .merge(cat_trans, on='product_category_name', how='left')
)

In [9]:
# Celula 9: validar que 1 linha = 1 order_id (sem explosao)
print(f'gold_raw shape: {gold_raw.shape}')
assert gold_raw['order_id'].is_unique, 'order_id deve ser unico na tabela gold'

gold_raw shape: (99441, 28)


In [10]:
# Celula 10: filtrar apenas pedidos validos para o modelo
# Excluir: canceled, unavailable e pedidos sem review (review_score nulo)
VALID_STATUS = ['delivered', 'shipped', 'invoiced', 'processing', 'approved']
gold_filtered = gold_raw[
    gold_raw['order_status'].isin(VALID_STATUS) &
    gold_raw['review_score'].notna() &
    gold_raw['order_approved_at'].notna()
].copy()

print(f'Antes do filtro: {gold_raw.shape[0]} | Apos filtro: {gold_filtered.shape[0]}')
print(f'Removidos: {gold_raw.shape[0] - gold_filtered.shape[0]}')

Antes do filtro: 99441 | Apos filtro: 97456
Removidos: 1985


## Secao 5: Join de Geolocation (Lat/Lon de Seller e Customer)

Enriquecer `gold_filtered` com coordenadas geograficas via CEP.  
**Correcao de dtype:** `seller_zip_code_prefix` vem como float64 apos o merge chain (NaN promove int->float).  
Converter via `Int64` nullable antes de `str.zfill(5)` para evitar `'9350.0'` ao inves de `'09350'`.

In [11]:
# Garantir dtype compativel antes do join
# NOTA: seller/customer zip vem como float64 (NaN no merge chain promove int->float)
# Converter via Int64 nullable para preservar zeros a esquerda: 9350.0 -> '09350'
gold_filtered = gold_filtered.copy()
gold_filtered['seller_zip_code_prefix'] = (
    gold_filtered['seller_zip_code_prefix'].astype('Int64').astype(str).str.zfill(5)
)
gold_filtered['customer_zip_code_prefix'] = (
    gold_filtered['customer_zip_code_prefix'].astype('Int64').astype(str).str.zfill(5)
)
geo_agg['geolocation_zip_code_prefix'] = (
    geo_agg['geolocation_zip_code_prefix'].astype(str).str.zfill(5)
)

print('Dtypes corrigidos para string zfill(5):')
print(f'  seller_zip sample: {gold_filtered["seller_zip_code_prefix"].head(3).tolist()}')
print(f'  customer_zip sample: {gold_filtered["customer_zip_code_prefix"].head(3).tolist()}')
print(f'  geo_agg zip sample: {geo_agg["geolocation_zip_code_prefix"].head(3).tolist()}')

Dtypes corrigidos para string zfill(5):
  seller_zip sample: ['09350', '31570', '14840']
  customer_zip sample: ['03149', '47813', '75265']
  geo_agg zip sample: ['01001', '01002', '01003']


In [12]:
# Celula 11: join de lat/lon do seller e customer
gold_with_geo = (
    gold_filtered
    .merge(
        geo_agg.rename(columns={
            'geolocation_zip_code_prefix': 'seller_zip_code_prefix',
            'geolocation_lat': 'seller_lat',
            'geolocation_lng': 'seller_lng',
        }),
        on='seller_zip_code_prefix',
        how='left',
    )
    .merge(
        geo_agg.rename(columns={
            'geolocation_zip_code_prefix': 'customer_zip_code_prefix',
            'geolocation_lat': 'customer_lat',
            'geolocation_lng': 'customer_lng',
        }),
        on='customer_zip_code_prefix',
        how='left',
    )
)

In [13]:
# Celula 12: validar que a tabela ainda tem 1 linha por order_id apos joins de geo
print(f'gold_with_geo shape: {gold_with_geo.shape}')
assert gold_with_geo['order_id'].is_unique, 'join de geo nao pode explodir cardinalidade'

gold_with_geo shape: (97456, 32)


In [14]:
# Celula 13: inspecionar CEPs sem lat/lon (CEPs nao presentes na tabela geo)
seller_sem_geo  = gold_with_geo['seller_lat'].isna().sum()
customer_sem_geo = gold_with_geo['customer_lat'].isna().sum()
print(f'Sellers sem lat/lon: {seller_sem_geo} ({seller_sem_geo/len(gold_with_geo):.1%})')
print(f'Customers sem lat/lon: {customer_sem_geo} ({customer_sem_geo/len(gold_with_geo):.1%})')
# Esperado: pequena proporcao de CEPs nao encontrados na geo table — documentar valor

Sellers sem lat/lon: 219 (0.2%)
Customers sem lat/lon: 272 (0.3%)


In [15]:
# Resumo final da tabela gold_with_geo
print('=== RESUMO FINAL ===')
print(f'gold_with_geo shape: {gold_with_geo.shape}')
print(f'order_id unico: {gold_with_geo["order_id"].is_unique}')
print(f'Colunas geo presentes: seller_lat={"seller_lat" in gold_with_geo.columns}, customer_lat={"customer_lat" in gold_with_geo.columns}')
print(f'\nColunas disponiveis ({len(gold_with_geo.columns)}):')
print(sorted(gold_with_geo.columns.tolist()))

=== RESUMO FINAL ===
gold_with_geo shape: (97456, 32)
order_id unico: True
Colunas geo presentes: seller_lat=True, customer_lat=True

Colunas disponiveis (32):
['customer_city', 'customer_id', 'customer_lat', 'customer_lng', 'customer_state', 'customer_unique_id', 'customer_zip_code_prefix', 'freight_value', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_id', 'order_purchase_timestamp', 'order_status', 'payment_value', 'product_category_name', 'product_category_name_english', 'product_height_cm', 'product_id', 'product_length_cm', 'product_photos_qty', 'product_weight_g', 'product_width_cm', 'review_score', 'seller_city', 'seller_id', 'seller_lat', 'seller_lng', 'seller_state', 'seller_zip_code_prefix', 'total_payment_value']
